# 第一阶段概念演示

这个 notebook 是为你的项目第一阶段准备的中文学习讲义。

我们会完成三件事：

1. 用更容易理解的话重新表述项目问题
2. 学懂 6 个最核心的机器学习基础概念
3. 通过一个很小但完整的教学实验，理解 `train/test split`、`accuracy` 和 `overfitting`


## 1. 项目问题是什么

项目的正式表述是：

> We compare tabular foundation models with boosted-tree baselines on small to medium tabular classification datasets.

把它翻成更适合新手理解的话，可以写成：

> 我们想研究，在中小型表格分类数据集上，现代的表格基础模型能不能达到或超过传统强基线树模型的效果。

你当前计划使用的模型包括：

- TabPFN v2
- TabICL
- XGBoost
- LightGBM

你当前初步选定的数据集包括：

- Adult
- Bank Marketing
- Credit-G

这里你先记住一件事：这个项目的重点不是“从零发明新模型”，而是“理解已有模型，并通过实验做出有根据的比较”。


## 2. 六个核心概念

### 2.1 classification

分类任务的目标是预测一个离散类别，也就是预测结果属于哪一类。

例子：

- 一个人的收入是否大于 `50K`
- 一个客户是否会订阅银行产品
- 一个申请人的信用风险是 `good` 还是 `bad`

如果模型输出的是一个类别标签，而不是一个连续数值，那么这通常就是分类任务。

### 2.2 train/test split

我们通常会把数据分成两部分：

- 训练集：让模型在这部分数据上学习规律
- 测试集：用来检查模型在没见过的新数据上表现如何

你可以把它理解成：

> 训练集是平时练习，测试集是正式考试。

如果不做这种划分，就很容易出现一种错觉：模型看起来很厉害，但它其实只是在记住原来的数据。

### 2.3 accuracy

准确率表示模型预测正确的比例。

公式是：

`accuracy = 预测正确的样本数 / 总样本数`

比如一共有 100 个样本，模型预测对了 82 个，那么准确率就是 `0.82`，也就是 `82%`。

### 2.4 overfitting

过拟合是指：模型在训练集上表现很好，但到了测试集上表现明显变差。

这通常意味着模型学到的不只是“规律”，还把训练数据中的细节甚至噪声也背下来了。

最常见的判断信号是：

- 训练集准确率非常高
- 测试集准确率没有同步提高，甚至更差

### 2.5 categorical features

类别特征指的是那些取值是类别，而不是连续数字的特征。

例如：

- 职业
- 学历
- 婚姻状态

这些特征在表格数据里非常常见。很多机器学习模型不能直接把字符串类别拿来算，所以往往需要先做编码或转换。

### 2.6 missing values

缺失值就是某些样本在某些列上没有值。

常见表现形式包括：

- 空白单元格
- `NaN`
- `?`
- `unknown`

真实世界的表格数据经常不完整，而不同模型对缺失值的处理方式可能很不一样，这会直接影响实验结果。


In [ ]:
from pathlib import Path

import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager

from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

plt.style.use('seaborn-v0_8')

candidate_font_names = [
    'Noto Sans CJK SC',
    'WenQuanYi Zen Hei',
    'SimHei',
    'Microsoft YaHei',
    'PingFang SC',
    'Arial Unicode MS',
]

windows_font_candidates = [
    Path('/mnt/c/Windows/Fonts/msyh.ttc'),
    Path('/mnt/c/Windows/Fonts/msyhbd.ttc'),
    Path('/mnt/c/Windows/Fonts/simhei.ttf'),
    Path('/mnt/c/Windows/Fonts/simsun.ttc'),
]

available_fonts = {font.name for font in font_manager.fontManager.ttflist}
selected_font = next((name for name in candidate_font_names if name in available_fonts), None)

if selected_font is None:
    for font_path in windows_font_candidates:
        if font_path.exists():
            font_manager.fontManager.addfont(str(font_path))
            selected_font = font_manager.FontProperties(fname=str(font_path)).get_name()
            break

if selected_font is not None:
    plt.rcParams['font.family'] = 'sans-serif'
    plt.rcParams['font.sans-serif'] = [selected_font]
else:
    print('未找到可用中文字体；请在 WSL 中执行: sudo apt install -y fonts-noto-cjk')

plt.rcParams['axes.unicode_minus'] = False


## 3. 先用一个最小例子理解 accuracy

在上真实数据集之前，我们先手算一次准确率。

假设：

- 真实标签是：`1 0 1 1 0`
- 模型预测是：`1 1 1 0 0`

其中预测正确的有 3 个，总共 5 个，所以准确率应该是 `3 / 5 = 0.6`。

下面我们分别用“手算”和 `sklearn` 来算一次，看结果是否一致。


In [ ]:
y_true = np.array([1, 0, 1, 1, 0])
y_pred = np.array([1, 1, 1, 0, 0])

manual_accuracy = (y_true == y_pred).sum() / len(y_true)
sklearn_accuracy = accuracy_score(y_true, y_pred)

print('手算 accuracy =', manual_accuracy)
print('sklearn 计算的 accuracy =', sklearn_accuracy)


## 4. 教学实验：一个真实的分类数据集

现在我们用 `scikit-learn` 自带的 `breast_cancer` 数据集来做一个教学实验。

为什么这里先用它？

- 它不需要额外下载
- 它体量小，适合新手第一次练习
- 它本身就是一个分类数据集
- 它可以帮助我们专注理解流程，而不是先被数据清洗卡住

注意：

这不是你最终项目里要用的 3 个正式数据集，它只是一个“练手用”的干净示例。


In [ ]:
X, y = load_breast_cancer(return_X_y=True, as_frame=True)

print('样本数 =', len(X))
print('特征数 =', X.shape[1])
print('\n类别计数：')
print(y.value_counts().sort_index())

X.head()


## 5. 什么是 train/test split

下面我们把数据划分成训练集和测试集。

这里用到的参数有：

- `test_size=0.2`：20% 的数据留作测试集
- `random_state=42`：保证你每次运行时切分结果可复现
- `stratify=y`：让训练集和测试集里的类别比例尽量保持一致

其中 `stratify=y` 在分类任务里很常见，因为如果类别分布不平衡，随便切分可能会让测试集偏掉，导致评估结果不稳定。


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print('训练集大小 =', len(X_train))
print('测试集大小 =', len(X_test))
print('\n训练集类别计数：')
print(y_train.value_counts().sort_index())
print('\n测试集类别计数：')
print(y_test.value_counts().sort_index())


## 6. 过拟合演示实验

现在我们比较两棵决策树：

- `深树`：默认深度，更灵活，也更容易把训练集记得很死
- `浅树`：`max_depth=3`，结构更简单，一般更不容易过拟合

我们会同时看：

- 训练集准确率
- 测试集准确率

如果某个模型训练集特别高，但测试集没有同样好，那就是典型的过拟合警报。


In [ ]:
models = {
    '深树': DecisionTreeClassifier(random_state=42),
    '浅树': DecisionTreeClassifier(max_depth=3, random_state=42),
}

rows = []

for name, model in models.items():
    start = time.perf_counter()
    model.fit(X_train, y_train)
    elapsed = time.perf_counter() - start
    train_acc = accuracy_score(y_train, model.predict(X_train))
    test_acc = accuracy_score(y_test, model.predict(X_test))
    rows.append({
        '模型': name,
        '训练集准确率': train_acc,
        '测试集准确率': test_acc,
        '训练耗时_秒': elapsed,
    })

results = pd.DataFrame(rows)
results


### 应该怎么读这张结果表

看到结果后，建议你先问自己 3 个问题：

1. 哪个模型的测试集准确率更高？
2. 训练集准确率和测试集准确率差得大不大？
3. 更复杂的模型，真的在“没见过的数据”上更好吗？

很多初学者一看到训练集很高就很开心，但机器学习里更重要的是泛化能力，也就是模型在测试集上的表现。


In [ ]:
plot_df = results.set_index('模型')[['训练集准确率', '测试集准确率']]

ax = plot_df.plot(kind='bar', figsize=(8, 4), rot=0)
ax.set_title('训练集与测试集准确率对比')
ax.set_ylabel('准确率')
ax.set_ylim(0.85, 1.02)
plt.show()


## 7. 为什么这能说明 overfitting

你运行后，大概率会看到类似这样的模式：

- 深树的训练集准确率非常高，甚至接近 1.0
- 但它的测试集准确率不一定最好
- 浅树的训练集准确率稍低一点
- 但它的测试集准确率可能反而更好

这说明什么？

这说明深树更容易把训练数据中的细节“记住”，而不是只学到稳定规律。所以一旦面对新样本，它未必表现更好。

这就是为什么我们不能只盯着训练集结果，更不能因为训练集是 `100%` 就觉得模型一定优秀。

这个结论在你后面做正式项目实验时会非常重要，因为你需要比较不同模型谁更能泛化，而不只是“谁更会背训练集”。


## 8. 类别特征和缺失值长什么样

你后面正式要做的是表格数据，而真实表格数据里经常会同时出现两类麻烦：

- 类别特征，比如职业、学历、地区
- 缺失值，比如空白、`NaN`、`unknown`

下面我们用一个很小的 toy dataframe 看一下它们长什么样。


In [ ]:
toy_df = pd.DataFrame({
    '职业': ['admin', 'technician', 'student', None],
    '学历': ['bachelor', 'master', None, 'high_school'],
    '年龄': [34, 41, 22, np.nan],
    '是否订阅': ['yes', 'no', 'yes', 'no'],
})

toy_df


In [ ]:
print('各列数据类型：')
print(toy_df.dtypes)

print('\n各列缺失值个数：')
print(toy_df.isna().sum())


你应该注意到：

- `职业` 和 `学历` 是类别特征
- `年龄` 是数值特征
- 有些单元格是缺失的

这已经比前面的练手数据更像真实项目数据了。

后面你真正开始做 Adult、Bank Marketing、Credit-G 时，我们就会进一步学：

- 怎么识别哪些列是类别特征
- 怎么统计每一列的缺失值
- 对 baseline 模型来说，预处理通常要做到什么程度


## 9. 为什么这些概念和你的项目直接相关

这些概念不是孤立的定义，它们会直接出现在你后面的项目实验里：

- `classification`：你当前项目版本先只做分类任务
- `train/test split`：公平比较不同模型时，必须使用一致的数据划分
- `accuracy`：这是你最核心的指标之一
- `overfitting`：它帮助你判断模型是在真正泛化，还是只是在训练集上表现好
- `categorical features`：Adult、Bank Marketing、Credit-G 里都会经常出现
- `missing values`：真实表格数据里很常见，也会影响模型表现和预处理策略


## 10. 自测问题

你可以先试着不用看答案，用自己的话回答下面 5 个问题：

1. 为什么你这个项目当前阶段先按 classification 做，而不是回归？
2. 为什么不能把所有数据都拿去训练，而是要保留测试集？
3. 如果一个模型 `训练集准确率 = 1.00`，但 `测试集准确率 = 0.91`，你会怀疑什么？
4. 为什么类别特征会让很多机器学习模型的处理变复杂？
5. 为什么缺失值在表格机器学习里不能忽略？


## 11. 下一步该做什么

当你把这个 notebook 从头运行并读完之后，我们的下一步就是：

1. 加载第一个真实的表格分类数据集
2. 看懂它的列、目标标签和数据规模
3. 先训练一个 baseline 模型，跑通完整流程

这就会是你真正意义上的第一个项目实验。
